## Preparo do ambiente

In [1]:
# Montando Drive
from google.colab import drive
drive.mount('/content/drive/')
%cd /content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph


In [2]:
# Instalando bibliotecas
!pip install torch_geometric
!pip install setproctitle

In [ ]:
import sys
print(sys.executable)

/usr/bin/python3


In [3]:
# Imports
from abregrafo import bizoia_resultados
from abregrafo import remove_onlysenders
import pickle
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data
from scipy.sparse import coo_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, ConfusionMatrixDisplay, auc, mean_squared_error
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from scipy.sparse.csgraph import connected_components
from scipy.sparse import lil_matrix
from sklearn.neighbors import NearestNeighbors

### Convertendo SAML-D

In [4]:
!ls

abregrafo.py	      History	       Models		    README.md
Analise_SAML_D.ipynb  imgs	       node_classification  requirements.txt
datasets	      LICENSE	       OpenGraph.ipynb	    Teste.ipynb
graph_generation      link_prediction  __pycache__	    testes


##### Leitura do arquivo

In [5]:
# SAML-D original (855.460 nós, 7.902 lavadores, 9.504.852 transações)
df = saml_d = pd.read_csv('../Datasets/SAML-D/SAML-D.csv')

In [ ]:
#laundering_types = df['Laundering_type'].unique()

In [6]:
laundering_types = [
    # --- Tipos Normais (Simulações de comportamento lícito) ---
    'Normal_Cash_Deposits',
    'Normal_Cash_Withdrawal',
    'Normal_Fan_In',
    'Normal_Fan_Out',
    'Normal_Foward',
    'Normal_Group',
    'Normal_Mutual',
    'Normal_Periodical',
    'Normal_Plus_Mutual',
    'Normal_Small_Fan_Out',
    'Normal_single_large',

    # --- Tipos de Lavagem (Tipologias e Técnicas Suspeitas) ---
    'Behavioural_Change_1',
    'Behavioural_Change_2',
    'Bipartite',
    'Cash_Withdrawal',  # Classificado como anômalo neste dataset
    'Cycle',
    'Deposit-Send',
    'Fan_In',           # Padrão de agrupamento suspeito
    'Fan_Out',          # Padrão de dispersão suspeito
    'Gather-Scatter',
    'Layered_Fan_In',
    'Layered_Fan_Out',
    'Over-Invoicing',
    'Scatter-Gather',
    'Single_large',
    'Smurfing',
    'Stacked Bipartite',
    'Structuring'
]

In [ ]:
# SAML-D mini (x nós, y lavadores, z transações)
df = saml_d_mini = pd.read_csv('../Datasets/SAML-D/SAML-D_mini.csv')

In [ ]:
# SAML-D 50% sem onlysenders (443381 nós, 2948 lavadores, 37797466 37.797.466 transações)
df = saml_d_50_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_mini2.csv')

In [ ]:
# SAML-D sem onlysenders (86.825.531 nós, 6.779 lavadores, 86.825.531 transações)
df = saml_d_nonlysenders = pd.read_csv('../Datasets/SAML-D/SAML-D_nonlysenders.csv')

In [ ]:
# SAML-D subgrafo bidirecional (66.370 nós, 5.120 lavadores, 783.735 transações)
df = saml_d_intersection = pd.read_csv('../Datasets/SAML-D/SAML-D_intersection.csv')
# Único funcionando!

---
##### Fim da leitura do arquivo

In [ ]:
df = saml_d

In [ ]:
df = saml_d_intersection

In [7]:
# Convertendo Date e Time em Timestamp Unix
df['timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time']).astype('int64')

In [8]:
# Criando mapeamento numérico para labels em string

currencies          = df.groupby('Payment_currency')['Payment_currency'].count().sort_values(ascending=False).keys()
bank_locations      = df.groupby('Sender_bank_location')['Sender_bank_location'].count().sort_values(ascending=False).keys()
payment_types       = df.groupby('Payment_type')['Payment_type'].count().sort_values(ascending=False).keys()
laundering_types    = df.groupby('Laundering_type')['Laundering_type'].count().sort_values(ascending=False).keys()
accounts            = pd.concat([df['Sender_account'], df['Receiver_account']]).unique()
launderers          = pd.concat([
                                  df[df['Is_laundering'] == 1]['Sender_account'],
                                  df[df['Is_laundering'] == 1]['Receiver_account']
                                ]).unique()
is_launderer        = pd.Series(accounts).isin(launderers).astype(int)

account_map         = {account:i for i, account in enumerate(accounts)}
currency_map        = {currencies[i]:i for i in range(len(currencies))}
bank_location_map   = {bank_locations[i]:i for i in range(len(bank_locations))}
payment_type_map    = {payment_types[i]:i for i in range(len(payment_types))}
laundering_type_map = {laundering_types[i]:i for i in range(len(laundering_types))}

In [9]:
Accounts = pd.DataFrame({
    'Accounts': accounts,
    'Launderers': pd.Series(accounts).isin(launderers).astype(int)
})
#Accounts

In [10]:
print(f'{'Accounts:':<12}{Accounts.shape[0]:>10}')
print(f'{'Lavadores:':<12}{ launderers.size:>10}')
print(f'{'Transações:':<12}{        df.shape[0]:>10}')

Accounts:       855460
Lavadores:        7902
Transações:    9504852


In [ ]:
df.columns

In [ ]:
payment_type_map

In [ ]:
# Criando labels sugeridas pelo chatGPT na conversa da Letícia

#1) Número de transações enviadas
sent_count = df.groupby('Sender_account').size()

#2) Número de transações recebidas
received_count = df.groupby('Receiver_account').size()

'''#3) Volume total já enviado por conta e moeda
sent_amount_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].sum().unstack(fill_value=0)
sent_amount_per_currency.columns = [f'sent_amount_{col.replace(' ', '_').lower()}' for col in sent_amount_per_currency.columns]

#4) Volume total já recebido por conta e moeda
received_amount_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].sum().unstack(fill_value=0)
received_amount_per_currency.columns = [f'received_amount_{col.replace(' ', '_').lower()}' for col in received_amount_per_currency.columns]
'''

#5) Valor médio já enviado por conta e moeda
sent_mean_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].mean().unstack(fill_value=0)
sent_mean_per_currency.columns = [f'sent_mean_{col.replace(' ', '_').lower()}' for col in sent_mean_per_currency.columns]

#6) Valor médio já recebido por conta e moeda
received_mean_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].mean().unstack(fill_value=0)
received_mean_per_currency.columns = [f'received_mean_{col.replace(' ', '_').lower()}' for col in received_mean_per_currency.columns]

'''#7) Maior valor enviado por conta e moeda
sent_max_per_currency = df.groupby(['Sender_account', 'Payment_currency'])['Amount'].max().unstack(fill_value=0)
sent_max_per_currency.columns = [f'sent_max_{col.replace(' ', '_').lower()}' for col in sent_max_per_currency.columns]

#8) Maior valor recebido por conta e moeda
received_max_per_currency = df.groupby(['Receiver_account', 'Payment_currency'])['Amount'].max().unstack(fill_value=0)
received_max_per_currency.columns = [f'received_max_{col.replace(' ', '_').lower()}' for col in received_max_per_currency.columns]
'''

#9) Percentual de reciprocidade dos vizinhos (para cada aresta de ida, quantas tem volta)
reciprocity_dict = {}
for account in accounts:
    outgoing_partners = set(df[df['Sender_account'] == account]['Receiver_account'].unique())
    incoming_partners = set(df[df['Receiver_account'] == account]['Sender_account'].unique())
    reciprocal_partners = outgoing_partners.intersection(incoming_partners)
    all_partners = outgoing_partners.union(incoming_partners)
    reciprocity_dict[account] = len(reciprocal_partners) / len(all_partners) if len(all_partners) > 0 else 0.0
reciprocity_score = pd.Series(reciprocity_dict, name='reciprocity_score')

#10) Histograma dos tipos de pagamento utilizados pelo nó
payment_type_sender_counts = df.groupby(['Sender_account', 'Payment_type']).size().unstack(fill_value=0)
payment_type_sender_counts.columns = [f'sent_payment_type_{col.replace(' ', '_').lower()}' for col in payment_type_sender_counts.columns]

#11) Total de moedas já utilizadas
sender_currencies = df.groupby('Sender_account')['Payment_currency'].apply(lambda x: set(x))
receiver_currencies = df.groupby('Receiver_account')['Received_currency'].apply(lambda x: set(x))
unique_currencies_per_account = {}
for acc in accounts:
    sent_c = sender_currencies.get(acc, set())
    rec_c = receiver_currencies.get(acc, set())
    unique_currencies_per_account[acc] = sent_c.union(rec_c)
currency_diversity = pd.Series({acc: len(currencies) for acc, currencies in unique_currencies_per_account.items()}, name='currency_diversity')

#12) Frequência de cada moeda utilizada
currency_sender_counts = df.groupby(['Sender_account', 'Payment_currency']).size().unstack(fill_value=0)
currency_sender_counts.columns = [f'sent_currency_{col.replace(' ', '_').lower()}' for col in currency_sender_counts.columns]
currency_receiver_counts = df.groupby(['Receiver_account', 'Received_currency']).size().unstack(fill_value=0)
currency_receiver_counts.columns = [f'received_currency_{col.replace(' ', '_').lower()}' for col in currency_receiver_counts.columns]
all_currency_counts_df = pd.DataFrame(index=accounts)
all_currency_counts_df = all_currency_counts_df.merge(currency_sender_counts, left_index=True, right_index=True, how='left').fillna(0)
all_currency_counts_df = all_currency_counts_df.merge(currency_receiver_counts, left_index=True, right_index=True, how='left').fillna(0)

'''# 5) Quantidade de moedas já utilizadas #### melhorar esse aqui (Implemented as part of #11, replacing the original line)

# 6) Quantidade de países (??) entender melhor esse depois
sender_bank_locations = df.groupby('Sender_account')['Sender_bank_location'].apply(lambda x: set(x))
receiver_bank_locations = df.groupby('Receiver_account')['Receiver_bank_location'].apply(lambda x: set(x))
unique_bank_locations_per_account = {}
for acc in accounts:
    sent_b = sender_bank_locations.get(acc, set())
    rec_b = receiver_bank_locations.get(acc, set())
    unique_bank_locations_per_account[acc] = sent_b.union(rec_b)
bank_diversity = pd.Series({acc: len(banks) for acc, banks in unique_bank_locations_per_account.items()}, name='bank_diversity')
'''

In [10]:
# Criando labels sugeridas pelo chatGPT na conversa da Letícia

#1) Número de transações enviadas
sent_count = df.groupby('Sender_account').size()

#2) Número de transações recebidas
received_count = df.groupby('Receiver_account').size()

'''#3) Volume total já enviado
sent_amount = df.groupby('Sender_account')['Amount'].sum()

#4) Volume total já recebido
received_amount = df.groupby('Receiver_account')['Amount'].sum()

#5) Valor médio enviado
sent_mean = df.groupby('Sender_account')['Amount'].mean()

#6) Valor médio recebido
received_mean = df.groupby('Receiver_account')['Amount'].mean()

#7) Maior valor enviado
sent_max = df.groupby('Sender_account')['Amount'].max()

#8) Maior valor recebido
received_max = df.groupby('Receiver_account')['Amount'].max()
'''

'''#9) Percentual de reciprocidade dos vizinhos (para cada aresta de ida, quantas tem volta)
reciprocity_dict = {}
for account in accounts:
    outgoing_partners = set(df[df['Sender_account'] == account]['Receiver_account'].unique())
    incoming_partners = set(df[df['Receiver_account'] == account]['Sender_account'].unique())
    reciprocal_partners = outgoing_partners.intersection(incoming_partners)
    all_partners = outgoing_partners.union(incoming_partners)
    reciprocity_dict[account] = len(reciprocal_partners) / len(all_partners) if len(all_partners) > 0 else 0.0
reciprocity_score = pd.Series(reciprocity_dict, name='reciprocity_score')
'''

#10) Histograma dos tipos de pagamento utilizados pelo nó
payment_type_sender_counts = df.groupby(['Sender_account', 'Payment_type']).size().unstack(fill_value=0)
payment_type_sender_counts.columns = [f'sent_payment_type_{col.replace(' ', '_').lower()}' for col in payment_type_sender_counts.columns]

#11) Total de moedas já utilizadas
sender_currencies = df.groupby('Sender_account')['Payment_currency'].apply(lambda x: set(x))
receiver_currencies = df.groupby('Receiver_account')['Received_currency'].apply(lambda x: set(x))
unique_currencies_per_account = {}
for acc in accounts:
    sent_c = sender_currencies.get(acc, set())
    rec_c = receiver_currencies.get(acc, set())
    unique_currencies_per_account[acc] = sent_c.union(rec_c)
currency_diversity = pd.Series({acc: len(currencies) for acc, currencies in unique_currencies_per_account.items()}, name='currency_diversity')

#12) Frequência de cada moeda utilizada
currency_sender_counts = df.groupby(['Sender_account', 'Payment_currency']).size().unstack(fill_value=0)
currency_sender_counts.columns = [f'sent_currency_{col.replace(' ', '_').lower()}' for col in currency_sender_counts.columns]
currency_receiver_counts = df.groupby(['Receiver_account', 'Received_currency']).size().unstack(fill_value=0)
currency_receiver_counts.columns = [f'received_currency_{col.replace(' ', '_').lower()}' for col in currency_receiver_counts.columns]
all_currency_counts_df = pd.DataFrame(index=accounts)
all_currency_counts_df = all_currency_counts_df.merge(currency_sender_counts, left_index=True, right_index=True, how='left').fillna(0)
all_currency_counts_df = all_currency_counts_df.merge(currency_receiver_counts, left_index=True, right_index=True, how='left').fillna(0)

'''# 5) Quantidade de moedas já utilizadas #### melhorar esse aqui (Implemented as part of #11, replacing the original line)

# 6) Quantidade de países (??) entender melhor esse depois
sender_bank_locations = df.groupby('Sender_account')['Sender_bank_location'].apply(lambda x: set(x))
receiver_bank_locations = df.groupby('Receiver_account')['Receiver_bank_location'].apply(lambda x: set(x))
unique_bank_locations_per_account = {}
for acc in accounts:
    sent_b = sender_bank_locations.get(acc, set())
    rec_b = receiver_bank_locations.get(acc, set())
    unique_bank_locations_per_account[acc] = sent_b.union(rec_b)
bank_diversity = pd.Series({acc: len(banks) for acc, banks in unique_bank_locations_per_account.items()}, name='bank_diversity')
'''

"# 5) Quantidade de moedas já utilizadas #### melhorar esse aqui (Implemented as part of #11, replacing the original line)\n\n# 6) Quantidade de países (??) entender melhor esse depois\nsender_bank_locations = df.groupby('Sender_account')['Sender_bank_location'].apply(lambda x: set(x))\nreceiver_bank_locations = df.groupby('Receiver_account')['Receiver_bank_location'].apply(lambda x: set(x))\nunique_bank_locations_per_account = {}\nfor acc in accounts:\n    sent_b = sender_bank_locations.get(acc, set())\n    rec_b = receiver_bank_locations.get(acc, set())\n    unique_bank_locations_per_account[acc] = sent_b.union(rec_b)\nbank_diversity = pd.Series({acc: len(banks) for acc, banks in unique_bank_locations_per_account.items()}, name='bank_diversity')\n"

## A Estudar

### Otimização das Features (Reciprocidade e Diversidade de Bancos)

Para melhorar a performance, vamos pré-calcular as informações necessárias para as features 'reciprocity_score' e 'bank_diversity' de forma mais eficiente, evitando a filtragem repetida do DataFrame dentro de loops.

In [11]:
# Otimização para a feature #9) Percentual de reciprocidade dos vizinhos

# 1. Pré-calcular todos os parceiros de envio e recebimento uma única vez
all_outgoing_partners = df.groupby('Sender_account')['Receiver_account'].apply(lambda x: set(x.unique()))
all_incoming_partners = df.groupby('Receiver_account')['Sender_account'].apply(lambda x: set(x.unique()))

reciprocity_data_optimized = {}
for account in accounts:
    # Usar .get() com um conjunto vazio padrão para contas que só enviam ou só recebem
    outgoing = all_outgoing_partners.get(account, set())
    incoming = all_incoming_partners.get(account, set())

    reciprocal_partners = outgoing.intersection(incoming)
    all_partners = outgoing.union(incoming) # Todos os parceiros únicos (enviados ou recebidos)
    reciprocity_data_optimized[account] = len(reciprocal_partners) / len(all_partners) if len(all_partners) > 0 else 0.0

reciprocity_score_optimized = pd.Series(reciprocity_data_optimized, name='reciprocity_score', index=accounts)


# Otimização para a feature #6) Quantidade de países (diversidade de bancos)

sender_bank_locations_sets = df.groupby('Sender_account')['Sender_bank_location'].apply(lambda x: set(x.unique()))
receiver_bank_locations_sets = df.groupby('Receiver_account')['Receiver_bank_location'].apply(lambda x: set(x.unique()))

bank_diversity_data_optimized = {}
for acc in accounts:
    sent_locs = sender_bank_locations_sets.get(acc, set())
    rec_locs = receiver_bank_locations_sets.get(acc, set())
    bank_diversity_data_optimized[acc] = len(sent_locs.union(rec_locs))

bank_diversity_optimized = pd.Series(bank_diversity_data_optimized, name='bank_diversity', index=accounts)

# Para usar estas novas series otimizadas, você substituiria as variáveis originais:
# reciprocity_score = reciprocity_score_optimized
# bank_diversity = bank_diversity_optimized


## A continuar

In [14]:
#Accounts = Accounts.merge(sent_amount.rename('sent_amount'), left_on='Accounts', right_index=True, how='left').fillna(0)
#Accounts = Accounts.merge(received_amount.rename('received_amount'), left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(sent_count.rename('sent_count'), left_on='Accounts', right_index=True, how='left').fillna(0)
Accounts = Accounts.merge(received_count.rename('received_count'), left_on='Accounts', right_index=True, how='left').fillna(0)
#Accounts = Accounts.merge(sent_currency_diversity, left_on='Accounts', right_index=True, how='left').fillna(0) # Updated sent_currency_diversity
Accounts = Accounts.merge(bank_diversity_optimized, left_on='Accounts', right_index=True, how='left').fillna(0) # New bank_diversity
Accounts = Accounts.merge(reciprocity_score_optimized, left_on='Accounts', right_index=True, how='left').fillna(0) # New reciprocity_score
Accounts = Accounts.merge(payment_type_sender_counts, left_on='Accounts', right_index=True, how='left').fillna(0) # New payment type histogram
Accounts = Accounts.merge(all_currency_counts_df, left_on='Accounts', right_index=True, how='left').fillna(0) # New currency frequency histogram

# Convertendo para inteiros as colunas de contagem e para float as de montante
Accounts['sent_count'] = Accounts['sent_count'].astype(int)
Accounts['received_count'] = Accounts['received_count'].astype(int)
#Accounts['sent_currency_diversity'] = Accounts['sent_currency_diversity'].astype(int)
Accounts['bank_diversity'] = Accounts['bank_diversity_optimized'].astype(int)
Accounts['reciprocity_score'] = Accounts['reciprocity_score_optimized'].astype(float)
#Accounts['sent_amount'] = Accounts['sent_amount'].astype(float)
#Accounts['received_amount'] = Accounts['received_amount'].astype(float)

# Cast the new payment type and currency frequency columns to int
for col in payment_type_sender_counts.columns:
    Accounts[col] = Accounts[col].astype(int)
for col in all_currency_counts_df.columns:
    Accounts[col] = Accounts[col].astype(int)

Accounts

,Accounts,Launderers,sent_count,received_count,bank_diversity,reciprocity_score,sent_payment_type_ach,sent_payment_type_cash_deposit,sent_payment_type_cash_withdrawal,sent_payment_type_cheque,...,received_currency_indian_rupee,received_currency_mexican_peso,received_currency_moroccan_dirham,received_currency_naira,received_currency_pakistani_rupee,received_currency_swiss_franc,received_currency_turkish_lira,received_currency_uk_pounds,received_currency_us_dollar,received_currency_yen
0,8724731955,0,15,0,1,0.000000,0,15,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1491989064,0,495,170,1,0.033333,89,0,40,107,...,0,0,0,0,0,0,0,170,0,0
2,287305149,0,221,271,1,0.125000,59,0,0,49,...,0,2,0,0,1,1,0,266,0,0
3,5376652437,0,9,0,1,0.000000,2,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,9614186178,0,6,0,1,0.000000,0,6,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
855455,5750497909,0,0,1,1,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
855456,351917955,0,0,1,1,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
855457,7236544862,0,0,1,1,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
855458,1927425677,0,0,1,1,0.000000,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [22]:
# Identificando colunas para normalização (todas exceto as protegidas)
cols_to_normalize = [col for col in Accounts.columns[2:]]

# Aplicando StandardScaler
scaler = StandardScaler()
Accounts[cols_to_normalize] = scaler.fit_transform(Accounts[cols_to_normalize])

# Verificando o resultado
Accounts

,Accounts,Launderers,sent_count,received_count,bank_diversity,reciprocity_score,sent_payment_type_ach,sent_payment_type_cash_deposit,sent_payment_type_cash_withdrawal,sent_payment_type_cheque,...,received_currency_indian_rupee,received_currency_mexican_peso,received_currency_moroccan_dirham,received_currency_naira,received_currency_pakistani_rupee,received_currency_swiss_franc,received_currency_turkish_lira,received_currency_uk_pounds,received_currency_us_dollar,received_currency_yen
0,8724731955,0,0.069751,-0.411197,-0.082983,-0.248610,-0.190812,8.822470,-0.106013,-0.190585,...,-0.03285,-0.031291,-0.030395,-0.031306,-0.03024,-0.031055,-0.03085,-0.395550,-0.030671,-0.031086
1,1491989064,0,8.678400,5.880294,-0.082983,-0.102822,7.041179,-0.157605,11.966797,8.482414,...,-0.03285,-0.031291,-0.030395,-0.031306,-0.03024,-0.031055,-0.03085,6.153443,-0.030671,-0.031086
2,287305149,0,3.764296,9.618180,-0.082983,0.298095,4.603429,-0.157605,-0.106013,3.781162,...,-0.03285,1.151706,-0.030395,-0.031306,0.53222,0.587759,-0.03085,9.851697,-0.030671,-0.031086
3,5376652437,0,-0.037857,-0.411197,-0.082983,-0.248610,-0.028296,-0.157605,-0.106013,-0.109529,...,-0.03285,-0.031291,-0.030395,-0.031306,-0.03024,-0.031055,-0.03085,-0.395550,-0.030671,-0.031086
4,9614186178,0,-0.091661,-0.411197,-0.082983,-0.248610,-0.190812,3.434425,-0.106013,-0.190585,...,-0.03285,-0.031291,-0.030395,-0.031306,-0.03024,-0.031055,-0.03085,-0.395550,-0.030671,-0.031086


### Transformações, formatações e manipulações

#### Cálculo de similaridade por cosseno

In [11]:
X = Accounts.drop(columns=['Accounts', 'Launderers']).values

nn = NearestNeighbors(n_neighbors=20, metric='cosine')
nn.fit(X)

distances, indices = nn.kneighbors(X)
similarity = 1 - distances

ValueError: Found array with 0 feature(s) (shape=(855460, 0)) while a minimum of 1 is required by NearestNeighbors.

In [ ]:
pd.DataFrame(similarity)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1.0,0.999939,0.999925,0.999912,0.999912,0.999900,0.999811,0.999786,0.999751,0.999748,0.999738,0.999734,0.999726,0.999720,0.999713,0.999711,0.999686,0.999633,0.999629,0.999612
1,1.0,0.995679,0.993080,0.992442,0.992435,0.991052,0.990479,0.990254,0.990224,0.989454,0.989376,0.988635,0.987565,0.987007,0.987007,0.986782,0.986203,0.985316,0.985110,0.985012
2,1.0,0.999239,0.999183,0.998928,0.998800,0.998510,0.998385,0.998232,0.998091,0.998011,0.997946,0.997915,0.997802,0.997799,0.997785,0.997746,0.997617,0.997615,0.997615,0.997577
3,1.0,0.999763,0.999722,0.999684,0.999608,0.999393,0.999339,0.999254,0.999119,0.999036,0.998960,0.998836,0.998809,0.998683,0.998641,0.998575,0.998571,0.998542,0.998449,0.998342
4,1.0,0.999799,0.999592,0.999462,0.999414,0.999334,0.999275,0.999270,0.999264,0.999249,0.999238,0.999188,0.999123,0.999055,0.998982,0.998926,0.998892,0.998866,0.998834,0.998829
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66365,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
66366,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
66367,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
66368,1.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [ ]:
similarity.size

1327400

In [ ]:
print(f"Number of elements less than 0.9: {(similarity == 1).sum()}")

Number of elements less than 0.9: 583275


In [ ]:
Accounts.columns

Index(['Accounts', 'Launderers', 'sent_count', 'received_count',
       'sent_payment_type_ach', 'sent_payment_type_cash_deposit',
       'sent_payment_type_cash_withdrawal', 'sent_payment_type_cheque',
       'sent_payment_type_credit_card', 'sent_payment_type_cross-border',
       'sent_payment_type_debit_card', 'sent_currency_albanian_lek',
       'sent_currency_dirham', 'sent_currency_euro',
       'sent_currency_indian_rupee', 'sent_currency_mexican_peso',
       'sent_currency_moroccan_dirham', 'sent_currency_naira',
       'sent_currency_pakistani_rupee', 'sent_currency_swiss_franc',
       'sent_currency_turkish_lira', 'sent_currency_uk_pounds',
       'sent_currency_us_dollar', 'sent_currency_yen',
       'received_currency_albanian_lek', 'received_currency_dirham',
       'received_currency_euro', 'received_currency_indian_rupee',
       'received_currency_mexican_peso', 'received_currency_moroccan_dirham',
       'received_currency_naira', 'received_currency_pakistani_rupee'

In [ ]:
# Normalização das features (testar e estudar dps)
#feats = np.log1p(feats)
#scaler = StandardScaler()
#feats = scaler.fit_transform(feats)

#### Conversão em PyTorch Data

In [ ]:
# Definindo edge_index e edge_attr

df_edge_index = pd.concat([
                            df['Sender_account'].map(account_map),
                            df['Receiver_account'].map(account_map)
                          ], axis=1)

df_edge_attr = pd.concat([
                  df[['timestamp', 'Amount']],
                  df['Payment_currency'].map(currency_map),
                  df['Received_currency'].map(currency_map),
                  df['Sender_bank_location'].map(bank_location_map),
                  df['Receiver_bank_location'].map(bank_location_map),
                  df['Payment_type'].map(payment_type_map),
                  df['Laundering_type'].map(laundering_type_map)
                ], axis=1)

In [ ]:
# Convertendo o dataset em Pytorch Data

data = Data(

    x= torch.ones((len(accounts),1)), # Label dos nós
    y= torch.tensor(df['Is_laundering'].values), # Label das arestas

    edge_index= torch.tensor(df_edge_index.values, dtype=torch.long),

    edge_attr= torch.tensor(df_edge_attr.values, dtype=torch.float)
)

data

#### Conversão em COO Matrix

In [24]:
# Convertendo para matriz esparsa

row = df['Sender_account'].map(account_map)
col = df['Receiver_account'].map(account_map)

values = np.ones(df.shape[0])

num_nodes = Accounts.shape[0]

adj = coo_matrix((values, (row, col)), shape=(num_nodes, num_nodes))
adj

<COOrdinate sparse matrix of dtype 'float64'
	with 9504852 stored elements and shape (855460, 855460)>

In [25]:
# Quantidade de Componentes
n_components, labels_cc = connected_components(adj)

print(n_components)

15592


Catapimbas, 15.592 componentes no SAML-D original.

No interseção, são 14.959. Não melhora muito.

#### Manipulação das componentes

In [26]:
# Lista de componentes (os nós que compõem a componente i)
components = [[] for _ in range(n_components)]
for node_index, component_label in enumerate(labels_cc):
    components[component_label].append(node_index)

In [31]:
adj = adj + adj.T
adj.data = np.ones_like(adj.data)

In [32]:
adj_lil = adj.tolil() # Formato que permite manipulação do coo_sparse

# Conectando (bidirecionalmente) componentes (peso desbalanceado, estudar)
for i in range(-1, n_components-1):
  adj_lil[ components[i][0], components[i+1][0] ] = 1.0
  adj_lil[ components[i+1][0], components[i][0] ] = 1.0

connected_adj = adj_lil.tocoo()
print("Número de componentes em connected_adj:", connected_components(connected_adj)[0] )

Número de componentes em connected_adj: 1


####

In [33]:
# Features dos nós (dummy, normalizado por coluna)
#feats = np.zeros((num_nodes, 8))
#feats = np.random.random(size=(num_nodes, 8))
#feats = feats / feats.sum(axis=0)
feats = Accounts.drop(columns=['Accounts', 'Launderers']).values
feats

array([[ 0.06975142, -0.41119738, -0.08298296, ..., -0.39554977,
        -0.03067091, -0.03108602],
       [ 8.67839963,  5.88029399, -0.08298296, ...,  6.15344276,
        -0.03067091, -0.03108602],
       [ 3.76429628,  9.61818004, -0.08298296, ...,  9.85169737,
        -0.03067091, -0.03108602],
       ...,
       [-0.19926883, -0.3741886 , -0.08298296, ..., -0.35702628,
        -0.03067091, -0.03108602],
       [-0.19926883, -0.3741886 , -0.08298296, ..., -0.39554977,
        -0.03067091, -0.03108602],
       [-0.19926883, -0.3741886 , -0.08298296, ..., -0.35702628,
        -0.03067091, -0.03108602]])

In [34]:
labels = np.array(is_launderer)
#labels = 1 - is_launderer.values

In [35]:
# Configurando máscaras... não entendi essa parte ainda

train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

# Proporcões não podem se sobrepor!
div1 = num_nodes*0.2
div2 = num_nodes*0.2 + div1
train_mask[:int(div1)] = True
val_mask[int(div1):int(div2)] = True
test_mask[int(div2):] = True
#test_mask[:] = True

In [36]:
# Gravando pickles

pickle.dump(adj, open('datasets/saml_d/adj_-1.pkl', 'wb'))
pickle.dump(feats, open('datasets/saml_d/feats.pkl', 'wb'))
pickle.dump(labels, open('datasets/saml_d/label.pkl', 'wb'))
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open('datasets/saml_d/mask_-1.pkl', 'wb'))

In [37]:
# Gravar apenas a máscara (testes)
pickle.dump({
    'train': train_mask,
    'valid': val_mask,
    'test': test_mask
}, open('datasets/saml_d/mask_-1.pkl', 'wb'))

In [38]:
!ls datasets/saml_d

adj_-1.pkl  feats.pkl  label.pkl  mask_-1.pkl


## Rodando o modelo

### Rodando o modelo

In [39]:
%cd node_classification

/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph/node_classification


In [40]:
!python main.py --epoch 0 --tstdata saml_d

2026-08-06 23:11:33.012849: Start
Dataset: saml_d, Node num: 855460, Edge num: 1681160
/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph/node_classification/data_handler.py:115: UserWarning: torch.sparse.SparseTensor(indices, values, shape, *, device=) is deprecated.  Please use torch.sparse_coo_tensor(indices, values, shape, dtype=, device=). (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:654.)
  asym_adj = t.sparse.FloatTensor(idxs, vals, shape)
Traceback (most recent call last):
  File "/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph/node_classification/main.py", line 197, in <module>
    multi_handler = MultiDataHandler(trn_datasets, tst_datasets)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph/node_classification/data_handler.py", line 21, in __init__
    handler = DataHandler(data_name, trn_flag, tst_flag)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  Fil

In [41]:
%cd ..

/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph


In [42]:
bizoia_resultados(1, Accounts.shape[0]) # 66.370

--- Resultado 0 ---


KeyboardInterrupt: 

## Git: Preparando para o Commit

Certifique-se de que o Git está configurado com suas informações e que você está no diretório correto do seu repositório.

In [43]:
# Verifique se o git está instalado e qual a versão
!git --version

# Confirme o diretório atual. Deve ser o seu repositório clonado.
!pwd


git version 2.34.1
/content/drive/MyDrive/Colab Notebooks/TCC_IC/OpenGraph


### 1. Configurar o Git (se ainda não o fez na sessão)

In [44]:
# Substitua 'Seu Nome' e 'seu.email@example.com' pelos seus dados
!git config --global user.name "Jon-Lemmon"
!git config --global user.email "luiz.gontijo@ufv.br"

### 2. Verificar o status do repositório

Use `git status` para ver quais arquivos foram modificados, adicionados ou excluídos.

In [ ]:
!git status

### 3. Adicionar arquivos para o commit

Use `git add` para adicionar os arquivos modificados ao "staging area".

-   `git add .` adiciona todos os arquivos novos e modificados no diretório atual.
-   `git add <nome_do_arquivo>` adiciona um arquivo específico.

In [ ]:
# Exemplo: Adicionar todos os arquivos modificados e novos
!git add .

# Verifique o status novamente para confirmar que os arquivos foram adicionados
!git status

### 4. Fazer o commit

Crie o commit com uma mensagem descritiva.

In [ ]:
# Substitua 'Sua mensagem de commit aqui' por uma descrição clara das suas alterações
!git commit -m "Sua mensagem de commit aqui"

### 5. Enviar as alterações para o GitHub (Push)

Agora, envie seus commits para o repositório remoto no GitHub.

**Autenticação:** O GitHub exige autenticação. Se você ativou `credential.helper store`, na primeira vez que você der push, será pedido seu nome de usuário do GitHub e um Personal Access Token (PAT). Crie um PAT no seu perfil do GitHub (Settings -> Developer settings -> Personal access tokens) com as permissões necessárias (geralmente `repo`).

In [ ]:
!git push origin main # Ou o nome da sua branch, como 'master'

Após o `git push` ser concluído com sucesso, suas alterações estarão no GitHub e você poderá continuar seu trabalho em outro ambiente.

### Lendo os resultados do modelo

In [ ]:
!ls

In [ ]:
with open('node_classification/Resultados/predicoes/predict0.pkl', 'rb') as f:
    data = pickle.load(f)

preds = data["preds"]
labels = data["labels"]
nodes = data['nodes']

In [ ]:
resultado = pd.DataFrame.from_dict({'preds': data["preds"], 'labels': data["labels"]})

In [ ]:
with open('node_classification/Resultados/embeddings/embedding0.pkl', 'rb') as f:
    embed = pickle.load(f)
embed

#### Testes testes

In [ ]:
s = 0
for e in embed:
    s += len(e)

print('Nós embeddados:', s)
print('Nós labelados:', len(labels))
print('Porcentagem de nós embeddados:', s/3327)

In [ ]:
resultado

In [ ]:
accuracy = accuracy_score(resultado['labels'], resultado['preds'])
precision = precision_score(resultado['labels'], resultado['preds'], average=None)
recall = recall_score(resultado['labels'], resultado['preds'], average=None)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

In [ ]:
labels_df = pd.DataFrame(labels, columns=['labels'])
labels_df

#### Testes 2

In [ ]:
resultado['preds'].value_counts()

In [ ]:
resultado['labels'].value_counts()

In [ ]:
comparison = (resultado['labels'] == labels_df['labels']).all()
print(f"All values match: {comparison}")

# Show detailed comparison
print("\nDetailed comparison:")
print((resultado['labels'] == labels_df['labels']).value_counts())

# Show any mismatches
mismatches = resultado[resultado['labels'] != labels_df['labels']]
if len(mismatches) > 0:
    print(f"\nMismatches found at indices:")
    print(mismatches)
else:
    print("\nNo mismatches found!")

In [ ]:
concat_emb = torch.cat([e.cpu() for e in embed], dim=0)
embed_df = pd.DataFrame(concat_emb.numpy())
embed_df

In [ ]:
for j in range(embed_df.shape[1]):
  if embed_df[j].nunique() > 1:
    print('coluna', j, 'tem', embed_df[j].nunique())

## Comparações com outros datasets

In [ ]:
with open('datasets/citeseer/feats.pkl', 'rb') as f:
    feats = pickle.load(f)
citeseer_feats_df = pd.DataFrame(feats)
citeseer_feats_df

In [ ]:
# Análise coluna a coluna
coluniques = [] # Conterá tuplas (n_coluna, qnt_uniques)
for j in range(citeseer_feats_df.shape[1]):
  if citeseer_feats_df[j].nunique() > 1:
    coluniques.append((j, citeseer_feats_df[j].nunique()))
print('Cada coluna possui pelo menos dois valores distintos?', len(coluniques) == citeseer_feats_df.shape[1])

In [ ]:
citeseer_feats_df[2].value_counts()

In [ ]:
with open('datasets/cora/feats.pkl', 'rb') as f:
    feats = pickle.load(f)
cora_feats_df = pd.DataFrame(feats)
cora_feats_df

In [ ]:
cora_feats_df[1].value_counts()

### Leitura dos Datasets Citeseer (código de IA)

Vamos ler os arquivos `citeseer.cites` e `citeseer.content` para entender sua estrutura.

In [ ]:
cites_path = '../Datasets/citeseer-doc-classification/citeseer.cites'
content_path = '../Datasets/citeseer-doc-classification/citeseer.content'

# Ler o arquivo citeseer.cites
# Este arquivo geralmente contém as relações de citação (arestas do grafo)
# A primeira coluna é o ID do documento citador, a segunda é o ID do documento citado.
cites_df = pd.read_csv(cites_path, sep='\t', header=None, names=['citing_paper_id', 'cited_paper_id'])

print('citeseer.cites (primeiras 5 linhas):')
display(cites_df.head())


O arquivo `citeseer.content` contém as características (features) de cada documento e seu rótulo (label).
A primeira coluna é o ID do documento, as colunas intermediárias são as features (geralmente representadas por termos de um vocabulário), e a última coluna é o rótulo da classe.

In [ ]:
# Ler o arquivo citeseer.content
# Este arquivo contém os IDs dos documentos, suas features e o label da classe.
# Como o número de features pode ser grande e não sabemos os nomes, vamos ler como DataFrame genérico.
content_df = pd.read_csv(content_path, sep='\t', header=None)

# A primeira coluna é o ID do documento
content_df.rename(columns={0: 'paper_id'}, inplace=True)

# A última coluna é o rótulo da classe
content_df.rename(columns={content_df.shape[1]-1: 'class_label'}, inplace=True)

print('\nciteseer.content (primeiras 5 linhas):')
display(content_df)

print('\nInformações do DataFrame content_df:')
display(content_df.info())


### Leitura dos Datasets PubMed (código de IA)

Vamos ler os arquivos `citeseer.cites` e `citeseer.content` para entender sua estrutura.

In [ ]:
cites_path = '../Datasets/citeseer-doc-classification/citeseer.cites'
content_path = '../Datasets/citeseer-doc-classification/citeseer.content'

# Ler o arquivo citeseer.cites
# Este arquivo geralmente contém as relações de citação (arestas do grafo)
# A primeira coluna é o ID do documento citador, a segunda é o ID do documento citado.
cites_df = pd.read_csv(cites_path, sep='\t', header=None, names=['citing_paper_id', 'cited_paper_id'])

print('citeseer.cites (primeiras 5 linhas):')
display(cites_df.head())


O arquivo `citeseer.content` contém as características (features) de cada documento e seu rótulo (label).
A primeira coluna é o ID do documento, as colunas intermediárias são as features (geralmente representadas por termos de um vocabulário), e a última coluna é o rótulo da classe.

In [ ]:
# Ler o arquivo citeseer.content
# Este arquivo contém os IDs dos documentos, suas features e o label da classe.
# Como o número de features pode ser grande e não sabemos os nomes, vamos ler como DataFrame genérico.
content_df = pd.read_csv(content_path, sep='\t', header=None)

# A primeira coluna é o ID do documento
content_df.rename(columns={0: 'paper_id'}, inplace=True)

# A última coluna é o rótulo da classe
content_df.rename(columns={content_df.shape[1]-1: 'class_label'}, inplace=True)

print('\nciteseer.content (primeiras 5 linhas):')
display(content_df)

print('\nInformações do DataFrame content_df:')
display(content_df.info())


### Voltando aqui

In [ ]:
# Comparando o citeseer original com o citeseer/feats.pkl


### Comparação Completa (com `citeseer_feats_df` ajustado)

Vamos agora comparar todos os documentos, ajustando o tamanho do `citeseer_feats_df` para que corresponda ao `content_df`.

In [ ]:
# 1. Extrair apenas as colunas de características de content_df
# As colunas de features estão entre 'paper_id' (coluna 0) e 'class_label' (última coluna)
content_features_df_full = content_df.iloc[:, 1:-1]

# 2. Ajustar o citeseer_feats_df para ter o mesmo número de linhas que content_features_df_full
num_rows_to_compare = len(content_features_df_full)
citeseer_feats_df_aligned = citeseer_feats_df.head(num_rows_to_compare)

print(f"Dimensões de content_features_df_full: {content_features_df_full.shape}")
print(f"Dimensões de citeseer_feats_df_aligned: {citeseer_feats_df_aligned.shape}")

# Assegurar que ambos os DataFrames têm a mesma forma para a comparação
if content_features_df_full.shape != citeseer_feats_df_aligned.shape:
    print("Erro: Os DataFrames não possuem as mesmas dimensões após o alinhamento.")
else:
    # 3. Realizar as comparações elemento a elemento

    # Condição 1: content_df == 0 e citeseer_feats_df != 0
    # Converter citeseer_feats_df_aligned para ter 0 onde é 0 e 1 onde é !=0 para melhor comparação
    # Ou, manter como está para ver a diferença exata
    condition1_matches_full = (content_features_df_full == 0) & (citeseer_feats_df_aligned != 0)
    count_condition1_full = condition1_matches_full.sum().sum()

    # Condição 2: content_df == 1 e citeseer_feats_df == 0
    condition2_matches_full = (content_features_df_full == 1) & (citeseer_feats_df_aligned == 0)
    count_condition2_full = condition2_matches_full.sum().sum()

    print(f"\nNúmero TOTAL de células onde content_df == 0 e citeseer_feats_df != 0: {count_condition1_full}")
    print(f"Número TOTAL de células onde content_df == 1 e citeseer_feats_df == 0: {count_condition2_full}")

    if count_condition1_full > 0 or count_condition2_full > 0:
        print("\nHá diferenças entre os datasets para as condições especificadas.")
        if count_condition1_full > 0:
            print("Exemplos de content_df == 0 e citeseer_feats_df != 0 (primeiros 5):")
            display(content_features_df_full[condition1_matches_full].stack().head())
            display(citeseer_feats_df_aligned[condition1_matches_full].stack().head())
        if count_condition2_full > 0:
            print("Exemplos de content_df == 1 e citeseer_feats_df == 0 (primeiros 5):")
            display(content_features_df_full[condition2_matches_full].stack().head())
            display(citeseer_feats_df_aligned[condition2_matches_full].stack().head())
    else:
        print("\nNenhuma diferença encontrada para as condições especificadas em todo o dataset alinhado.")


### Contagem de Zeros e Não-Zeros nos DataFrames de Características

Vamos verificar a distribuição de valores (zeros e não-zeros) em ambos os DataFrames alinhados para ter uma visão mais ampla da sua correspondência.

In [ ]:
# Para content_features_df_full
zeros_content_full = (content_features_df_full == 0).sum().sum()
non_zeros_content_full = (content_features_df_full != 0).sum().sum()
total_elements_content_full = content_features_df_full.size

print(f"--- content_features_df_full ---")
print(f"Total de elementos: {total_elements_content_full}")
print(f"Total de zeros: {zeros_content_full}")
print(f"Total de não-zeros: {non_zeros_content_full}")
print(f"Porcentagem de zeros: {zeros_content_full / total_elements_content_full:.2%}")
print(f"Porcentagem de não-zeros: {non_zeros_content_full / total_elements_content_full:.2%}")

print(f"\n--- citeseer_feats_df_aligned ---")
# Para citeseer_feats_df_aligned
zeros_citeseer_aligned = (citeseer_feats_df_aligned == 0).sum().sum()
non_zeros_citeseer_aligned = (citeseer_feats_df_aligned != 0).sum().sum()
total_elements_citeseer_aligned = citeseer_feats_df_aligned.size

print(f"Total de elementos: {total_elements_citeseer_aligned}")
print(f"Total de zeros: {zeros_citeseer_aligned}")
print(f"Total de não-zeros: {non_zeros_citeseer_aligned}")
print(f"Porcentagem de zeros: {zeros_citeseer_aligned / total_elements_citeseer_aligned:.2%}")
print(f"Porcentagem de não-zeros: {non_zeros_citeseer_aligned / total_elements_citeseer_aligned:.2%}")

# Comparação direta dos totais
print("\n--- Comparação dos Totais ---")
print(f"Zeros correspondem: {zeros_content_full == zeros_citeseer_aligned}")
print(f"Não-zeros correspondem: {non_zeros_content_full == non_zeros_citeseer_aligned}")


## Testando outros Classificadores

Primeiro, é feita a divisão de treinamento. Depois, basta treinar um dos classificadores, e então, rodar as métricas.

In [ ]:
# Divisão de treinamento
train_x, test_x, train_y, test_y = train_test_split(embed_df, labels_df['labels'], test_size=0.4, random_state=18, stratify=labels_df['labels'])
train_y = np.ravel(train_y)
test_y = np.ravel(test_y)

#### Random Forest

In [ ]:
classificador_randomforest = RandomForestClassifier(n_estimators=200, random_state=18, max_depth=4, class_weight='balanced')
classificador_randomforest.fit(train_x, train_y)
pred_y = classificador_randomforest.predict(test_x)

#### XGBoost

In [ ]:
# embed_df possui a embedding dos 325888 nós
# is_launderer possui as classificações (0 ou 1) para os nós fraudadores e não fraudadores
scale_pos_weight = (labels_df[labels_df['labels'] == 1].size/labels_df[labels_df['labels'] == 0].size)**(1/2)

In [ ]:
classificador_xgb = xgb.XGBClassifier(objective='binary:logistic', random_state=18, scale_pos_weight=scale_pos_weight)
classificador_xgb.fit(train_x, train_y)
pred_y = classificador_xgb.predict(test_x)

### Métricas

In [ ]:
# Resultado Random Forest
accuracy = accuracy_score(test_y, pred_y)
precision = precision_score(test_y, pred_y)
recall = recall_score(test_y, pred_y)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

In [ ]:
# Resultado XGBoost
accuracy = accuracy_score(test_y, pred_y)
precision = precision_score(test_y, pred_y)
recall = recall_score(test_y, pred_y)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)

# Playground
---
Testes temporários ou desorganizados

In [ ]:
!ls

### Abrindo pikles

In [ ]:
%cd ..

In [ ]:
!ls

In [ ]:
with open('datasets/citeseer/adj_-1.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

In [ ]:
with open('datasets/pubmed/feats.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

In [ ]:
with open('datasets/saml_d/label.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

In [ ]:
is_launderer = pd.DataFrame(data)
is_launderer

In [ ]:
len(data)

In [ ]:
with open('datasets/saml_d/mask_-1.pkl', 'rb') as f:
    data = pickle.load(f)
print(data)

In [ ]:
for v in ['train', 'valid', 'test']:
  for b_1, b1, b5 in zip(data_n1[v], data_1[v], data_5[v]):
    if not (b_1 == b1 and b1 == b5):
      print(b_1, b1, b5)
  print()

In [ ]:
with open('graph_generation/gen_results/datasets/gen_data_ecommerce/embedding_dict.pkl', 'rb') as f:
    data = pickle.load(f)
data

#### Escrevendo Pikles

In [ ]:
# Armazenando pkl para leitura no OpenGraph

with open('datasets/saml_d/saml_d_sparse_graph.pkl', 'wb') as f:
    pickle.dump(A, f)

graph_data = {
    'adj': A,
    'edge_index': data.edge_index,
    'edge_attr': data.edge_attr,
    'y': data.y
}

with open('datasets/saml_d/saml_d_sparse_graph.pkl', 'wb') as f:
    pickle.dump(graph_data, f)

In [ ]:
!ls